In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# การตั้งค่าเริ่มต้นและตัวเลือกการกำหนดค่าของ Transpiler


<details>
<summary><b>เวอร์ชันของแพคเกจ</b></summary>

โค้ดในหน้านี้พัฒนาขึ้นโดยใช้ข้อกำหนดต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Abstract circuits ต้องถูก transpile เนื่องจาก QPU มีชุด basis gates ที่จำกัดและไม่สามารถรันการดำเนินการได้ทุกอย่าง ฟังก์ชันของ Transpiler คือการเปลี่ยนวงจรที่หลากหลายให้สามารถรันบน QPU ที่ระบุได้ ซึ่งทำโดยการแปลงวงจรเป็น basis gates ที่รองรับ และเพิ่ม SWAP gates ตามความจำเป็น เพื่อให้การเชื่อมต่อของวงจรตรงกับ QPU

ดังที่อธิบายไว้ใน [Transpile ด้วย pass managers](transpile-with-pass-managers) คุณสามารถสร้าง [pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) โดยใช้ฟังก์ชัน [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) และส่งวงจรหรือรายการวงจรไปยังเมธอด [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) เพื่อ transpile วงจรเหล่านั้น คุณสามารถเรียก `generate_preset_pass_manager` โดยส่งแค่ระดับการปรับแต่งและ Backend เพื่อใช้ค่าเริ่มต้นสำหรับตัวเลือกอื่นทั้งหมด หรือจะส่ง argument เพิ่มเติมเพื่อปรับแต่ง transpilation ได้

## การใช้งานเบื้องต้นโดยไม่มีพารามิเตอร์
ในตัวอย่างนี้ เราส่งวงจรและ QPU เป้าหมายไปยัง Transpiler โดยไม่ระบุพารามิเตอร์เพิ่มเติม

สร้างวงจรและดูผลลัพธ์:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

Transpile วงจรและดูผลลัพธ์:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## พารามิเตอร์ทั้งหมดที่ใช้ได้
ต่อไปนี้คือพารามิเตอร์ทั้งหมดที่ใช้ได้สำหรับฟังก์ชัน [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) มีสอง class ของ argument คือ class ที่อธิบายเป้าหมายของการ compile และ class ที่มีผลต่อวิธีการทำงานของ Transpiler

พารามิเตอร์ทั้งหมดยกเว้น `optimization_level` เป็นตัวเลือก สำหรับรายละเอียดครบถ้วน ดู [เอกสาร API ของ Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api)

- `optimization_level` (int) - ระดับการปรับแต่งที่จะดำเนินการกับวงจร เป็น integer ในช่วง (0 - 3) ระดับที่สูงกว่าจะสร้างวงจรที่ปรับแต่งมากขึ้น แต่ใช้เวลา transpilation นานกว่า ดู [การตั้งค่าระดับการปรับแต่งของ Transpiler](set-optimization) สำหรับรายละเอียดเพิ่มเติม

### พารามิเตอร์สำหรับอธิบายเป้าหมายของการ compile:
argument เหล่านี้อธิบาย QPU เป้าหมายสำหรับการรันวงจร รวมถึงข้อมูลเช่น coupling map ของ QPU (ซึ่งอธิบายการเชื่อมต่อของ Qubit) basis gates ที่ QPU รองรับ และอัตราข้อผิดพลาดของ Gates

พารามิเตอร์เหล่านี้หลายตัวถูกอธิบายอย่างละเอียดใน [พารามิเตอร์ที่ใช้บ่อยสำหรับ transpilation](common-parameters)

<details>
  <summary>
    **พารามิเตอร์ QPU (`Backend`)**
  </summary>

**พารามิเตอร์ Backend** - ถ้าคุณระบุ `backend` คุณไม่จำเป็นต้องระบุ `target` หรือตัวเลือก backend อื่นใด ในทำนองเดียวกัน ถ้าคุณระบุ `target` คุณไม่จำเป็นต้องระบุ `backend` หรือตัวเลือก backend อื่นใด
- `backend` (Backend) - ถ้าตั้งค่านี้ Transpiler จะ compile วงจร input ไปยังอุปกรณ์นี้ ถ้ามีตัวเลือกอื่นที่ส่งผลต่อการตั้งค่าเหล่านี้ เช่น `coupling_map` มันจะ override การตั้งค่าจาก `backend`
- `target` (Target) - เป้าหมาย Transpiler ของ Backend โดยปกติจะระบุเป็นส่วนหนึ่งของ argument ของ backend แต่ถ้าคุณสร้าง Target object เอง คุณสามารถระบุที่นี่ได้ ซึ่งจะ override เป้าหมายจาก `backend`
- `backend_properties` (BackendProperties) - คุณสมบัติที่ QPU ส่งกลับมา รวมถึงข้อมูลเกี่ยวกับข้อผิดพลาดของ Gate ข้อผิดพลาดในการอ่านค่า เวลา coherence ของ Qubit และอื่นๆ หา QPU ที่ให้ข้อมูลนี้โดยรัน `backend.properties()`
- `timing_constraints` (Dict[str, int] | None) - ข้อจำกัดฮาร์ดแวร์ control แบบ optional เกี่ยวกับความละเอียดของเวลา instruction ข้อมูลนี้ได้รับจากการกำหนดค่า QPU ถ้า QPU ไม่มีข้อจำกัดใดๆ เกี่ยวกับการจัดสรรเวลา instruction `timing_constraints` จะเป็น `None` และไม่มีการปรับใด QPU อาจรายงานชุดข้อจำกัด ได้แก่:
    - `granularity`: ค่า integer แทนความละเอียดขั้นต่ำของ pulse gate ในหน่วย dt pulse gate ที่ผู้ใช้กำหนดควรมีระยะเวลาที่เป็นผลคูณของค่า granularity นี้
    - `min_length`: ค่า integer แทนความยาวขั้นต่ำของ pulse gate ในหน่วย dt pulse gate ที่ผู้ใช้กำหนดควรยาวกว่าความยาวนี้
    - `pulse_alignment`: ค่า integer แทนความละเอียดของเวลาของเวลาเริ่มต้น instruction ของ Gate instruction ของ Gate ควรเริ่มที่เวลาที่เป็นผลคูณของค่านี้
    - `acquire_alignment`: ค่า integer แทนความละเอียดของเวลาของเวลาเริ่มต้น instruction การวัด instruction การวัดควรเริ่มที่เวลาที่เป็นผลคูณของค่านี้
</details>

<details>
  <summary>
    **พารามิเตอร์ Layout และ topology**
  </summary>

- `basis_gates` (List[str] | None) - รายการชื่อ basis gate สำหรับ unroll ตัวอย่างเช่น ['u1', 'u2', 'u3', 'cx'] ถ้าเป็น `None` จะไม่ทำการ unroll
- `coupling_map` (CouplingMap | List[List[int]]) - Directed coupling map (อาจเป็นแบบกำหนดเอง) สำหรับเป้าหมายใน mapping ถ้า coupling map มีความสมมาตร ต้องระบุทั้งสองทิศทาง รองรับรูปแบบเหล่านี้:
    - CouplingMap instance
    - List - ต้องให้เป็น adjacency matrix โดยแต่ละ entry ระบุ two-qubit interactions ที่ directed ทั้งหมดที่ QPU รองรับ ตัวอย่างเช่น: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - การ mapping ของ circuit operations ไปยัง pulse schedules ถ้าเป็น `None` จะใช้ `instruction_schedule_map` ของ QPU
</details>

### พารามิเตอร์สำหรับมีผลต่อวิธีการทำงานของ Transpiler
พารามิเตอร์เหล่านี้ส่งผลต่อขั้นตอน transpilation เฉพาะ บางตัวอาจส่งผลต่อหลายขั้นตอน แต่ถูกระบุไว้ภายใต้ขั้นตอนเดียวเพื่อความง่าย ถ้าคุณระบุ argument เช่น `initial_layout` สำหรับ Qubit ที่ต้องการใช้ ค่านั้นจะ override pass ทั้งหมดที่อาจเปลี่ยนมัน กล่าวคือ Transpiler จะไม่เปลี่ยนสิ่งที่คุณระบุด้วยตนเอง สำหรับรายละเอียดเกี่ยวกับขั้นตอนเฉพาะ ดู [ขั้นตอนของ Transpiler](transpiler-stages)

<details>
  <summary>
    **ขั้นตอน Initialization**
  </summary>

- `hls_config` (HLSConfig) - class การกำหนดค่า `HLSConfig` แบบ optional ที่ส่งต่อโดยตรงไปยัง transformation pass `HighLevelSynthesis` class การกำหนดค่านี้ช่วยให้คุณระบุรายการ synthesis algorithms และพารามิเตอร์สำหรับ high-level objects ต่างๆ
- `init_method` (str) - ชื่อ plugin สำหรับใช้ในขั้นตอน initialization โดยค่าเริ่มต้นไม่ใช้ external plugin คุณดูรายการ plugins ที่ติดตั้งได้โดยรัน `list_stage_plugins()` พร้อม `init` สำหรับ argument ชื่อ stage
- `unitary_synthesis_method` (str) - ชื่อ unitary synthesis method ที่จะใช้ โดยค่าเริ่มต้นจะใช้ `default` คุณดูรายการ plugins ที่ติดตั้งได้โดยรัน `unitary_synthesis_plugin_names()`
- `unitary_synthesis_plugin_config` (dict) - dictionary การกำหนดค่าแบบ optional ที่ส่งต่อโดยตรงไปยัง unitary synthesis plugin โดยค่าเริ่มต้นการตั้งค่านี้ไม่มีผลเนื่องจาก unitary synthesis method เริ่มต้นไม่รับการกำหนดค่าแบบกำหนดเอง การใช้การกำหนดค่าแบบกำหนดเองจำเป็นเฉพาะเมื่อระบุ unitary synthesis plugin ด้วย argument `unitary_synthesis` เนื่องจากสิ่งนี้เป็นแบบกำหนดเองสำหรับแต่ละ unitary synthesis plugin ดูเอกสารของ plugin สำหรับวิธีใช้ตัวเลือกนี้
</details>

<details>
  <summary>
    **ขั้นตอน Layout**
  </summary>

- `initial_layout` (Layout | Dict | List) - ตำแหน่งเริ่มต้นของ virtual qubits บน physical qubits ถ้า layout นี้ทำให้วงจรเข้ากันได้กับข้อจำกัด `coupling_map` จะถูกใช้ layout สุดท้ายไม่ได้รับประกันว่าจะเหมือนกัน เนื่องจาก Transpiler อาจสลับ qubits ผ่าน swaps หรือวิธีอื่น สำหรับรายละเอียดครบถ้วน ดู [ส่วน Initial layout](common-parameters#initial-layout)
- `layout_method` (str) - ชื่อ layout selection pass (`default`, `dense`, `sabre` และ `trivial`) นี้ยังสามารถเป็นชื่อ external plugin สำหรับใช้ในขั้นตอน layout คุณดูรายการ plugins ที่ติดตั้งได้โดยรัน `list_stage_plugins()` พร้อม `layout` สำหรับ argument `stage_name` ค่าเริ่มต้นคือ `sabre`
</details>

<details>
  <summary>
    **ขั้นตอน Routing**
  </summary>

- `routing_method` (str) - ชื่อ routing pass (`basic`, `lookahead`, `default`, `sabre` หรือ `none`) นี้ยังสามารถเป็นชื่อ external plugin สำหรับใช้ในขั้นตอน routing คุณดูรายการ plugins ที่ติดตั้งได้โดยรัน `list_stage_plugins()` พร้อม `routing` สำหรับ argument `stage_name` ค่าเริ่มต้นคือ `sabre`
</details>

<details>
  <summary>
    **ขั้นตอน Translation**
  </summary>

- `translation_method` (str) - ชื่อ translation pass (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`) นี้ยังสามารถเป็นชื่อ external plugin สำหรับใช้ในขั้นตอน translation คุณดูรายการ plugins ที่ติดตั้งได้โดยรัน `list_stage_plugins()` พร้อม `translation` สำหรับ argument `stage_name` ค่าเริ่มต้นคือ `translator`
</details>

<details>
  <summary>
    **ขั้นตอน Optimization**
  </summary>

- `approximation_degree` (float ในช่วง 0-1 | None) - dial แบบ heuristic ที่ใช้สำหรับการประมาณวงจร (1.0 = ไม่มีการประมาณ, 0.0 = การประมาณสูงสุด) ค่าเริ่มต้นคือ 1.0 การระบุ `None` จะตั้งค่าระดับการประมาณเป็นอัตราข้อผิดพลาดที่รายงาน ดู [ส่วน Approximation degree](common-parameters#approx-degree) สำหรับรายละเอียดเพิ่มเติม
- `optimization_method` (str) - ชื่อ plugin สำหรับใช้ในขั้นตอน optimization โดยค่าเริ่มต้นไม่ใช้ external plugin คุณดูรายการ plugins ที่ติดตั้งได้โดยรัน `list_stage_plugins()` พร้อม `optimization` สำหรับ argument `stage_name`
</details>

<details>
  <summary>
    **ขั้นตอน Scheduling**
  </summary>

- `scheduling_method` (str) - ชื่อ scheduling pass นี้ยังสามารถเป็นชื่อ external plugin สำหรับใช้ในขั้นตอน scheduling คุณดูรายการ plugins ที่ติดตั้งได้โดยรัน `list_stage_plugins()` พร้อม `scheduling` สำหรับ argument `stage_name`
  * 'as_soon_as_possible': กำหนดเวลา instructions อย่าง greedy โดยเร็วที่สุดเท่าที่เป็นไปได้บน qubit resource (alias: `asap`)
  * 'as_late_as_possible': กำหนดเวลา instructions ช้า กล่าวคือ รักษา qubits ในสถานะพื้นฐานเมื่อเป็นไปได้ (alias: `alap`) นี่คือค่าเริ่มต้น
</details>

<details>
  <summary>
    **การรัน Transpiler**
  </summary>

- `seed_transpiler` (int) - ตั้งค่า random seeds สำหรับส่วนที่เป็น stochastic ของ Transpiler
</details>

ค่าเริ่มต้นต่อไปนี้จะถูกใช้ถ้าคุณไม่ระบุพารามิเตอร์ใดข้างต้น ดู[หน้า API reference](../api/qiskit/transpiler_preset) ของเมธอดสำหรับข้อมูลเพิ่มเติม:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>